# HouseDiffusion → TarkeebAI plan_schema (Phase 2)

Run HouseDiffusion on a free Colab/Kaggle **GPU** and get a `plan_schema.json` your Revit executor can build — driven by a **bubble diagram** (rooms + adjacencies), not raw text.

Pipeline: `bubble diagram` → `graph_to_model_input` → **HouseDiffusion** → `model_output_to_plan` → `plan_schema.json`.

> The orchestrating LLM (Claude / Antigravity) writes the bubble diagram from the user's brief. This notebook only runs the model — the same `housediffusion` bridge package used by the `plan_from_bubble_diagram` MCP tool.

**Runtime → Change runtime type → GPU (T4)** before you start.

In [ ]:
!nvidia-smi -L  # confirm a GPU is attached (T4 is plenty)

## 1. Get the code
The sampling path only needs `torch` + `numpy` (already on Colab) plus `gdown`/`jsonschema`. We do **not** install HouseDiffusion's old pinned `requirements.txt` (TensorFlow / mpi4py / nightly torch) — the bridge adds the source to `sys.path` and never imports the dataset/FID code that needs them.

In [ ]:
!git clone https://github.com/mhmdTaqi-code/PlanForgeRevit.git
%cd PlanForgeRevit
!pip -q install gdown jsonschema

## 2. Download a checkpoint
The authors' temporary RPLAN checkpoint (from the HouseDiffusion README). Replace the id with your own fine-tuned checkpoint later (Phase 3).

In [ ]:
import os, gdown
os.makedirs('ckpts', exist_ok=True)
CKPT = 'ckpts/model.pt'
if not os.path.exists(CKPT):
    # HouseDiffusion temporary model: https://github.com/aminshabani/house_diffusion#readme
    gdown.download(id='16zKmtxwY5lF6JE-CJGkRf3-OFoD1TrdR', output=CKPT, quiet=False)

os.environ['HOUSE_DIFFUSION_SRC'] = os.path.abspath('AIProjects/house_diffusion')
os.environ['HOUSE_DIFFUSION_CKPT'] = os.path.abspath(CKPT)
print('checkpoint:', os.environ['HOUSE_DIFFUSION_CKPT'])

## 3. Describe the house as a bubble diagram
This is what the orchestrating LLM produces from a natural-language brief. `type` uses the plan-schema vocabulary; `adjacencies` are pairs of room ids that share a door/wall. See `schema/bubble_diagram_schema.json`.

In [ ]:
diagram = {
    'meta': {'name': 'Iraqi 10x20 house',
             'plot': {'width_m': 10, 'depth_m': 20},
             'description': 'guest majlis near the entrance, family living, kitchen, 3 bedrooms'},
    'rooms': [
        {'id': 'maj',  'type': 'majlis'},
        {'id': 'liv',  'type': 'family_living'},
        {'id': 'kit',  'type': 'kitchen'},
        {'id': 'corr', 'type': 'corridor'},
        {'id': 'bed1', 'type': 'master_bedroom'},
        {'id': 'bed2', 'type': 'bedroom'},
        {'id': 'bed3', 'type': 'bedroom'},
        {'id': 'bath', 'type': 'bathroom'},
    ],
    'adjacencies': [
        ['maj', 'corr'], ['liv', 'corr'], ['kit', 'liv'],
        ['corr', 'bed1'], ['corr', 'bed2'], ['corr', 'bed3'], ['bed1', 'bath'],
    ],
}

## 4. Generate → plan_schema.json

In [ ]:
import sys, json
sys.path.insert(0, 'mcp-servers/plan-generator')
from housediffusion import engine

print('engine status:', engine.availability())  # should be ready=True now
plan = engine.generate_from_diagram(diagram, num_samples=1)

with open('plan_schema.json', 'w', encoding='utf-8') as f:
    json.dump(plan, f, ensure_ascii=False, indent=2)
print(json.dumps(plan, ensure_ascii=False, indent=2))

In [ ]:
# Validate against the shared schema (same check the MCP validate_plan tool runs)
import json, jsonschema
schema = json.load(open('schema/plan_schema.json', encoding='utf-8'))
errs = list(jsonschema.Draft202012Validator(schema).iter_errors(plan))
print('schema validation:', 'OK' if not errs else [e.message for e in errs])

## 5. Next
Download `plan_schema.json` (Files pane) and hand it to your Revit executor — exactly like a `generate_plan` result. The orchestration prompt in `docs/connect-mcp-clients.md` converts meters → millimeters and builds the walls.

**Quality tips:** if rooms overlap or look off, sample a few (`num_samples=8`) and pick the best, adjust `adjacencies`, or set `corners` per room. HouseDiffusion is RPLAN-trained (foreign typology) — Phase 3 fine-tunes it on Iraqi/Gulf plans.